# Diabetes.ipynb


In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import plotly.express as px
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
import xgboost as xbg
import os 
import sys

In [2]:
cwd = os.getcwd()

print(cwd)

/Users/saikeerthan/NYP-AI/Year3/AI_Application/Diabetes


In [3]:
df = os.path.join(cwd, "data/diabetes_dataset00.csv")

if df:
    print("YES")
else: 
    print("NO")

YES


In [4]:
df = pd.read_csv(df)

df

,Target,Genetic Markers,Autoantibodies,Family History,Environmental Factors,Insulin Levels,Age,BMI,Physical Activity,Dietary Habits,...,Pulmonary Function,Cystic Fibrosis Diagnosis,Steroid Use History,Genetic Testing,Neurological Assessments,Liver Function Tests,Digestive Enzyme Levels,Urine Test,Birth Weight,Early Onset Symptoms
0,Steroid-Induced Diabetes,Positive,Negative,No,Present,40,44,38,High,Healthy,...,76,No,No,Positive,3,Normal,56,Ketones Present,2629,No
1,Neonatal Diabetes Mellitus (NDM),Positive,Negative,No,Present,13,1,17,High,Healthy,...,60,Yes,No,Negative,1,Normal,28,Glucose Present,1881,Yes
2,Prediabetic,Positive,Positive,Yes,Present,27,36,24,High,Unhealthy,...,80,Yes,No,Negative,1,Abnormal,55,Ketones Present,3622,Yes
3,Type 1 Diabetes,Negative,Positive,No,Present,8,7,16,Low,Unhealthy,...,89,Yes,No,Positive,2,Abnormal,60,Ketones Present,3542,No
4,Wolfram Syndrome,Negative,Negative,Yes,Present,17,10,17,High,Healthy,...,41,No,No,Positive,1,Normal,24,Protein Present,1770,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,Steroid-Induced Diabetes,Negative,Negative,Yes,Present,24,38,35,High,Unhealthy,...,71,Yes,Yes,Positive,2,Abnormal,34,Normal,2575,No
69996,LADA,Positive,Positive,No,Absent,21,51,31,Low,Unhealthy,...,82,Yes,Yes,Negative,1,Abnormal,58,Ketones Present,3002,No
69997,Type 1 Diabetes,Positive,Negative,No,Absent,18,11,15,Low,Unhealthy,...,77,Yes,Yes,Positive,2,Normal,53,Protein Present,3593,No
69998,Cystic Fibrosis-Related Diabetes (CFRD),Positive,Negative,No,Absent,32,30,24,High,Healthy,...,70,No,No,Positive,1,Abnormal,35,Ketones Present,2592,Yes


In [5]:
df.isnull().sum()

Target                           0
Genetic Markers                  0
Autoantibodies                   0
Family History                   0
Environmental Factors            0
Insulin Levels                   0
Age                              0
BMI                              0
Physical Activity                0
Dietary Habits                   0
Blood Pressure                   0
Cholesterol Levels               0
Waist Circumference              0
Blood Glucose Levels             0
Ethnicity                        0
Socioeconomic Factors            0
Smoking Status                   0
Alcohol Consumption              0
Glucose Tolerance Test           0
History of PCOS                  0
Previous Gestational Diabetes    0
Pregnancy History                0
Weight Gain During Pregnancy     0
Pancreatic Health                0
Pulmonary Function               0
Cystic Fibrosis Diagnosis        0
Steroid Use History              0
Genetic Testing                  0
Neurological Assessm

In [6]:
# print a summary of each column 


executive_summary = pd.DataFrame({
    "columns": df.columns,
    "num_unique": [df[c].nunique(dropna=False) for c in df.columns],
    "samples": [df[c].unique()[:5] for c in df.columns]
})

In [7]:
executive_summary

,columns,num_unique,samples
0,Target,13,"[Steroid-Induced Diabetes, Neonatal Diabetes M..."
1,Genetic Markers,2,"[Positive, Negative]"
2,Autoantibodies,2,"[Negative, Positive]"
3,Family History,2,"[No, Yes]"
4,Environmental Factors,2,"[Present, Absent]"
5,Insulin Levels,45,"[40, 13, 27, 8, 17]"
6,Age,80,"[44, 1, 36, 7, 10]"
7,BMI,28,"[38, 17, 24, 16, 26]"
8,Physical Activity,3,"[High, Low, Moderate]"
9,Dietary Habits,2,"[Healthy, Unhealthy]"


In [8]:
# List of placeholders to check for
placeholders = ["unknown", "Unknown", "UNK", "Not Available", "N/A", "na", "?", "-", "missing"]

# Create a summary dictionary
unknown_summary = {}

for col in df.columns:
    count = 0
    for placeholder in placeholders:
        count += (df[col].astype(str).str.strip().str.lower() == placeholder.lower()).sum()
    if count > 0:
        unknown_summary[col] = count

# Show summary
if unknown_summary:
    print("Columns with unknown values and their counts:")
    for col, cnt in unknown_summary.items():
        print(f"{col}: {cnt}")
else:
    print("No 'unknown' values found in any column!")

No 'unknown' values found in any column!


In [9]:
# Separate columns by type based on your summary
numerical_cols = [
    'Insulin Levels', 'Age', 'BMI', 'Blood Pressure', 'Cholesterol Levels',
    'Waist Circumference', 'Blood Glucose Levels', 'Birth Weight',
    'Weight Gain During Pregnancy', 'Pancreatic Health', 'Pulmonary Function',
    'Digestive Enzyme Levels'
]
# If you want, you can add more columns with many unique values

print("=== Outlier Summary by Column ===\n")
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
    print(f"{col}: {len(outliers)} outliers")
    # Optionally: print extreme values
    # print(outliers.sort_values().head(3))
    # print(outliers.sort_values(ascending=False).head(3))
print("\n=== Done ===")

=== Outlier Summary by Column ===

Insulin Levels: 0 outliers
Age: 0 outliers
BMI: 0 outliers
Blood Pressure: 0 outliers
Cholesterol Levels: 0 outliers
Waist Circumference: 522 outliers
Blood Glucose Levels: 0 outliers
Birth Weight: 0 outliers
Weight Gain During Pregnancy: 0 outliers
Pancreatic Health: 0 outliers
Pulmonary Function: 1206 outliers
Digestive Enzyme Levels: 0 outliers

=== Done ===


In [10]:
categorical_cols = [
    'Genetic Markers', 'Autoantibodies', 'Family History', 'Environmental Factors', 
    'Physical Activity', 'Dietary Habits', 'Ethnicity', 'Socioeconomic Factors', 
    'Smoking Status', 'Alcohol Consumption', 'Glucose Tolerance Test', 
    'History of PCOS', 'Previous Gestational Diabetes', 'Pregnancy History', 
    'Cystic Fibrosis Diagnosis', 'Steroid Use History', 'Genetic Testing', 
    'Neurological Assessments', 'Liver Function Tests', 'Urine Test', 
    'Early Onset Symptoms'
]
for col in categorical_cols:
    print(f"\n{col} value counts:")
    print(df[col].value_counts())



Genetic Markers value counts:
Genetic Markers
Positive    35101
Negative    34899
Name: count, dtype: int64

Autoantibodies value counts:
Autoantibodies
Negative    35058
Positive    34942
Name: count, dtype: int64

Family History value counts:
Family History
Yes    35168
No     34832
Name: count, dtype: int64

Environmental Factors value counts:
Environmental Factors
Absent     35088
Present    34912
Name: count, dtype: int64

Physical Activity value counts:
Physical Activity
Moderate    23427
Low         23348
High        23225
Name: count, dtype: int64

Dietary Habits value counts:
Dietary Habits
Healthy      35020
Unhealthy    34980
Name: count, dtype: int64

Ethnicity value counts:
Ethnicity
Low Risk     35018
High Risk    34982
Name: count, dtype: int64

Socioeconomic Factors value counts:
Socioeconomic Factors
Medium    23413
High      23304
Low       23283
Name: count, dtype: int64

Smoking Status value counts:
Smoking Status
Smoker        35045
Non-Smoker    34955
Name: count

In [11]:
for col in ["Waist Circumference", "Pulmonary Function"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    print(f"\n{col} - min: {df[col].min()}, max: {df[col].max()}, lower: {lower}, upper: {upper}")
    print(df[col].describe())



Waist Circumference - min: 20, max: 54, lower: 16.5, upper: 52.5
count    70000.000000
mean        35.051657
std          6.803461
min         20.000000
25%         30.000000
50%         34.000000
75%         39.000000
max         54.000000
Name: Waist Circumference, dtype: float64

Pulmonary Function - min: 30, max: 89, lower: 39.0, upper: 103.0
count    70000.000000
mean        70.264671
std         11.965600
min         30.000000
25%         63.000000
50%         72.000000
75%         79.000000
max         89.000000
Name: Pulmonary Function, dtype: float64


In [12]:
# Calculate value counts for the target variable
class_counts = df['Target'].value_counts().sort_values(ascending=False)

# Plotly bar plot
fig = px.bar(
    x=class_counts.index,
    y=class_counts.values,
    labels={'x': 'Diabetes Type', 'y': 'Count'},
    title='Distribution of Diabetes Types in Dataset',
    text=class_counts.values
)
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_tickangle=30)
fig.show()


In [13]:
# Plot percentage distribution with Plotly
class_percent = 100 * class_counts / class_counts.sum()
fig = px.bar(
    x=class_percent.index,
    y=class_percent.values,
    labels={'x': 'Diabetes Type', 'y': 'Percentage (%)'},
    title='Percentage Distribution of Diabetes Types in Dataset',
    text=class_percent.round(2).astype(str) + '%'
)
fig.update_traces(textposition='inside')
fig.update_layout(xaxis_tickangle=30)
fig.show()


# ML Modelling

In [14]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

## XGBOOST

### First Round of Training

In [15]:
# 2. Define features
target_col = 'Target'
categorical_cols = [
    'Genetic Markers', 'Autoantibodies', 'Family History', 'Environmental Factors', 
    'Physical Activity', 'Dietary Habits', 'Ethnicity', 'Socioeconomic Factors', 
    'Smoking Status', 'Alcohol Consumption', 'Glucose Tolerance Test', 
    'History of PCOS', 'Previous Gestational Diabetes', 'Pregnancy History', 
    'Cystic Fibrosis Diagnosis', 'Steroid Use History', 'Genetic Testing', 
    'Neurological Assessments', 'Liver Function Tests', 'Urine Test', 'Early Onset Symptoms'
]
numerical_cols = [
    'Insulin Levels', 'Age', 'BMI', 'Blood Pressure', 'Cholesterol Levels',
    'Waist Circumference', 'Blood Glucose Levels', 'Birth Weight',
    'Weight Gain During Pregnancy', 'Pancreatic Health', 'Pulmonary Function',
    'Digestive Enzyme Levels'
]
# Drop columns if you find any non-predictive (e.g., IDs)

In [16]:
# 3. Encode target variable
y = df[target_col].astype(str)
y_cat, y_labels = pd.factorize(y)  # Saves mapping for inverse transform

# 4. Encode categorical variables (OneHotEncoder, handle_unknown ensures no error)
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat = ohe.fit_transform(df[categorical_cols])

# 5. Combine with numerical features
X_num = df[numerical_cols].values
X = np.hstack([X_num, X_cat])

# 6. Train/val/test split (70/15/15 split)
X_temp, X_test, y_temp, y_test = train_test_split(X, y_cat, test_size=0.15, random_state=42, stratify=y_cat)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)
# 0.1765 * 0.85 ≈ 0.15 (so you get ~70/15/15 split)

In [17]:
# 7. Build and train XGBoost model (multiclass)
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=8,
    objective='multi:softprob',
    num_class=len(y_labels),
    eval_metric='mlogloss',
    use_label_encoder=False,
    n_jobs=-1,
    random_state=42
)
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=True)

[0]	validation_0-mlogloss:2.08555
[1]	validation_0-mlogloss:1.80658
[2]	validation_0-mlogloss:1.60261
[3]	validation_0-mlogloss:1.44197


/Users/saikeerthan/NYP-AI/Year3/new_y3s1/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning:

[20:11:50] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.




[4]	validation_0-mlogloss:1.31007
[5]	validation_0-mlogloss:1.19838
[6]	validation_0-mlogloss:1.10197
[7]	validation_0-mlogloss:1.01791
[8]	validation_0-mlogloss:0.94446
[9]	validation_0-mlogloss:0.87904
[10]	validation_0-mlogloss:0.82079
[11]	validation_0-mlogloss:0.76854
[12]	validation_0-mlogloss:0.72183
[13]	validation_0-mlogloss:0.67939
[14]	validation_0-mlogloss:0.64103
[15]	validation_0-mlogloss:0.60617
[16]	validation_0-mlogloss:0.57446
[17]	validation_0-mlogloss:0.54564
[18]	validation_0-mlogloss:0.51922
[19]	validation_0-mlogloss:0.49523
[20]	validation_0-mlogloss:0.47340
[21]	validation_0-mlogloss:0.45332
[22]	validation_0-mlogloss:0.43484
[23]	validation_0-mlogloss:0.41797
[24]	validation_0-mlogloss:0.40225
[25]	validation_0-mlogloss:0.38788
[26]	validation_0-mlogloss:0.37464
[27]	validation_0-mlogloss:0.36237
[28]	validation_0-mlogloss:0.35108
[29]	validation_0-mlogloss:0.34061
[30]	validation_0-mlogloss:0.33098
[31]	validation_0-mlogloss:0.32206
[32]	validation_0-mlogloss

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=-1, num_class=13, ...)

In [18]:
# 8. Evaluate on validation and test sets
def print_metrics(y_true, y_pred, y_prob, set_name="Set"):
    print(f"\n=== {set_name} Metrics ===")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Macro F1-score:", f1_score(y_true, y_pred, average='macro'))
    print("Per-class report:\n", classification_report(y_true, y_pred, target_names=y_labels))

print_metrics(y_val, xgb.predict(X_val), xgb.predict_proba(X_val), "Validation")
print_metrics(y_test, xgb.predict(X_test), xgb.predict_proba(X_test), "Test")


=== Validation Metrics ===
Accuracy: 0.9059226813940202
Macro F1-score: 0.9050763863220934
Per-class report:
                                             precision    recall  f1-score   support

                  Steroid-Induced Diabetes       0.82      0.82      0.82       791
          Neonatal Diabetes Mellitus (NDM)       1.00      1.00      1.00       811
                               Prediabetic       0.98      1.00      0.99       807
                           Type 1 Diabetes       0.88      0.96      0.92       817
                          Wolfram Syndrome       0.88      0.94      0.91       797
                                      LADA       0.97      0.95      0.96       784
                           Type 2 Diabetes       0.86      0.73      0.79       810
                 Wolcott-Rallison Syndrome       0.94      0.87      0.90       810
                        Secondary Diabetes       0.82      0.76      0.79       822
Type 3c Diabetes (Pancreatogenic Diabetes)      

In [19]:
from sklearn.metrics import confusion_matrix
import plotly.figure_factory as ff

# Generate confusion matrix
cm = confusion_matrix(y_test, xgb.predict(X_test))

# Convert to DataFrame for nice display
cm_df = pd.DataFrame(cm, index=y_labels, columns=y_labels)

# Plot with Plotly
fig = ff.create_annotated_heatmap(
    z=cm_df.values,
    x=cm_df.columns.tolist(),
    y=cm_df.index.tolist(),
    colorscale='Blues',
    showscale=True,
    annotation_text=cm_df.values,
    hoverinfo='z'
)

fig.update_layout(
    title='Confusion Matrix (Test Set)',
    xaxis_title='Predicted Label',
    yaxis_title='True Label',
    autosize=False,
    width=900,
    height=900,
    margin=dict(l=200, r=50, b=150, t=100)
)
fig.update_xaxes(side="bottom")
fig.show()

## LIGHTGBM

In [20]:
import lightgbm as lgb
# LightGBM needs the same X, y as before
lgbm = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=8,
    objective='multiclass',
    num_class=len(y_labels),
    random_state=42,
    n_jobs=-1
)

lgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)])

# Evaluate
print("LightGBM Validation:")
print_metrics(y_val, lgbm.predict(X_val), None, "Validation")
print("LightGBM Test:")
print_metrics(y_test, lgbm.predict(X_test), None, "Test")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001734 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1338
[LightGBM] [Info] Number of data points in the train set: 48998, number of used features: 60
[LightGBM] [Info] Start training from score -2.585340
[LightGBM] [Info] Start training from score -2.560469
[LightGBM] [Info] Start training from score -2.566563
[LightGBM] [Info] Start training from score -2.553625
[LightGBM] [Info] Start training from score -2.577787
[LightGBM] [Info] Start training from score -2.595410
[LightGBM] [Info] Start training from score -2.562849
[LightGBM] [Info] Start training from score -2.562055
[LightGBM] [Info] Start training from score -2.547610
[LightGBM] [Info] Start training from score -2.576981
[LightGBM] [Info] Start training from score -2.572694
[LightGBM] [Info] Start training from score -2.550482

/Users/saikeerthan/NYP-AI/Year3/new_y3s1/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names




=== Validation Metrics ===
Accuracy: 0.9044943820224719
Macro F1-score: 0.9036770001301822
Per-class report:
                                             precision    recall  f1-score   support

                  Steroid-Induced Diabetes       0.83      0.81      0.82       791
          Neonatal Diabetes Mellitus (NDM)       1.00      1.00      1.00       811
                               Prediabetic       0.98      1.00      0.99       807
                           Type 1 Diabetes       0.89      0.95      0.92       817
                          Wolfram Syndrome       0.88      0.93      0.91       797
                                      LADA       0.95      0.95      0.95       784
                           Type 2 Diabetes       0.85      0.74      0.79       810
                 Wolcott-Rallison Syndrome       0.93      0.87      0.90       810
                        Secondary Diabetes       0.82      0.76      0.79       822
Type 3c Diabetes (Pancreatogenic Diabetes)      

/Users/saikeerthan/NYP-AI/Year3/new_y3s1/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names




=== Test Metrics ===
Accuracy: 0.8998095238095238
Macro F1-score: 0.8990044471233567
Per-class report:
                                             precision    recall  f1-score   support

                  Steroid-Induced Diabetes       0.82      0.81      0.81       791
          Neonatal Diabetes Mellitus (NDM)       1.00      1.00      1.00       811
                               Prediabetic       0.96      1.00      0.98       806
                           Type 1 Diabetes       0.88      0.94      0.91       817
                          Wolfram Syndrome       0.90      0.94      0.92       797
                                      LADA       0.95      0.93      0.94       783
                           Type 2 Diabetes       0.84      0.73      0.78       810
                 Wolcott-Rallison Syndrome       0.94      0.89      0.92       810
                        Secondary Diabetes       0.81      0.75      0.77       822
Type 3c Diabetes (Pancreatogenic Diabetes)       0.81 

## RandomForest

In [21]:
# 1. Build the Random Forest model
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=16,
    class_weight='balanced',   # Handles any mild class imbalance
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# 2. Fit the model
rf.fit(X_train, y_train)

# 3. Evaluate on validation and test sets
def print_metrics_rf(y_true, y_pred, set_name="Set"):
    print(f"\n=== {set_name} Metrics (Random Forest) ===")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Macro F1-score:", f1_score(y_true, y_pred, average='macro'))
    print("Per-class report:\n", classification_report(y_true, y_pred, target_names=y_labels))

print_metrics_rf(y_val, rf.predict(X_val), "Validation")
print_metrics_rf(y_test, rf.predict(X_test), "Test")


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    0.6s
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:    3.0s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:    3.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done 200 out of 200 | elapsed:    0.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    0.1s



=== Validation Metrics (Random Forest) ===
Accuracy: 0.8967815654161112
Macro F1-score: 0.8955762856595051
Per-class report:
                                             precision    recall  f1-score   support

                  Steroid-Induced Diabetes       0.80      0.82      0.81       791
          Neonatal Diabetes Mellitus (NDM)       1.00      1.00      1.00       811
                               Prediabetic       0.93      1.00      0.96       807
                           Type 1 Diabetes       0.85      1.00      0.92       817
                          Wolfram Syndrome       0.85      1.00      0.92       797
                                      LADA       0.93      0.95      0.94       784
                           Type 2 Diabetes       0.90      0.72      0.80       810
                 Wolcott-Rallison Syndrome       1.00      0.83      0.91       810
                        Secondary Diabetes       0.82      0.75      0.78       822
Type 3c Diabetes (Pancreatogenic

[Parallel(n_jobs=8)]: Done 200 out of 200 | elapsed:    0.1s finished


# Deep Learning

In [22]:
import tensorflow as tf 
from tensorflow import keras

In [23]:
# 2. Column definitions
target_col = 'Target'
categorical_cols = [
    'Genetic Markers', 'Autoantibodies', 'Family History', 'Environmental Factors', 
    'Physical Activity', 'Dietary Habits', 'Ethnicity', 'Socioeconomic Factors', 
    'Smoking Status', 'Alcohol Consumption', 'Glucose Tolerance Test', 
    'History of PCOS', 'Previous Gestational Diabetes', 'Pregnancy History', 
    'Cystic Fibrosis Diagnosis', 'Steroid Use History', 'Genetic Testing', 
    'Neurological Assessments', 'Liver Function Tests', 'Urine Test', 'Early Onset Symptoms'
]
numerical_cols = [
    'Insulin Levels', 'Age', 'BMI', 'Blood Pressure', 'Cholesterol Levels',
    'Waist Circumference', 'Blood Glucose Levels', 'Birth Weight',
    'Weight Gain During Pregnancy', 'Pancreatic Health', 'Pulmonary Function',
    'Digestive Enzyme Levels'
]

In [24]:
# 3. Encode target
y = df[target_col].astype(str)
y_cat, y_labels = pd.factorize(y)  # y_labels saves the mapping
num_classes = len(y_labels)

# 4. Preprocessing - Encoding
# -- Categorical features: OneHot
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat = ohe.fit_transform(df[categorical_cols])

# -- Numerical features: StandardScaler
scaler = StandardScaler()
X_num = scaler.fit_transform(df[numerical_cols])

# -- Concatenate
X = np.hstack([X_num, X_cat])


In [25]:
# 5. Train/val/test split (same as before)
X_temp, X_test, y_temp, y_test = train_test_split(X, y_cat, test_size=0.15, random_state=42, stratify=y_cat)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)

# 6. To categorical (one-hot targets for Keras)
y_train_oh = keras.utils.to_categorical(y_train, num_classes)
y_val_oh   = keras.utils.to_categorical(y_val, num_classes)
y_test_oh  = keras.utils.to_categorical(y_test, num_classes)

In [26]:
model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(num_classes, activation='softmax')
])

In [27]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │        31,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 13)             │         1,677 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 200,717 (784.05 KB)

 Trainable params: 198,925 (777.05 KB)

 Non-trainable params: 1,792 (7.00 KB)

In [28]:
# 8. Train model
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(
    X_train, y_train_oh,
    epochs=50,
    batch_size=1024,
    validation_data=(X_val, y_val_oh),
    callbacks=[early_stop],
    verbose=2
)


Epoch 1/50
48/48 - 2s - 44ms/step - accuracy: 0.5446 - loss: 1.2731 - val_accuracy: 0.7035 - val_loss: 1.1174
Epoch 2/50
48/48 - 1s - 17ms/step - accuracy: 0.7049 - loss: 0.7357 - val_accuracy: 0.6896 - val_loss: 0.9149
Epoch 3/50
48/48 - 1s - 20ms/step - accuracy: 0.7277 - loss: 0.6759 - val_accuracy: 0.7010 - val_loss: 0.8053
Epoch 4/50
48/48 - 1s - 19ms/step - accuracy: 0.7449 - loss: 0.6281 - val_accuracy: 0.7199 - val_loss: 0.7256
Epoch 5/50
48/48 - 1s - 19ms/step - accuracy: 0.7571 - loss: 0.5998 - val_accuracy: 0.7566 - val_loss: 0.6388
Epoch 6/50
48/48 - 1s - 19ms/step - accuracy: 0.7684 - loss: 0.5756 - val_accuracy: 0.7673 - val_loss: 0.5946
Epoch 7/50
48/48 - 1s - 19ms/step - accuracy: 0.7746 - loss: 0.5557 - val_accuracy: 0.7764 - val_loss: 0.5635
Epoch 8/50
48/48 - 1s - 19ms/step - accuracy: 0.7811 - loss: 0.5371 - val_accuracy: 0.7908 - val_loss: 0.5257
Epoch 9/50
48/48 - 1s - 20ms/step - accuracy: 0.7898 - loss: 0.5214 - val_accuracy: 0.8053 - val_loss: 0.4948
Epoch 10/5

In [29]:
# 9. Evaluate
def print_dl_metrics(y_true, y_pred, set_name="Set"):
    print(f"\n=== {set_name} Metrics (Deep Learning) ===")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Macro F1-score:", f1_score(y_true, y_pred, average='macro'))
    print("Per-class report:\n", classification_report(y_true, y_pred, target_names=y_labels))

y_val_pred = np.argmax(model.predict(X_val), axis=1)
y_test_pred = np.argmax(model.predict(X_test), axis=1)

print_dl_metrics(y_val, y_val_pred, "Validation")
print_dl_metrics(y_test, y_test_pred, "Test")

329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 816us/step

=== Validation Metrics (Deep Learning) ===
Accuracy: 0.8539325842696629
Macro F1-score: 0.8532013884484113
Per-class report:
                                             precision    recall  f1-score   support

                  Steroid-Induced Diabetes       0.81      0.71      0.76       791
          Neonatal Diabetes Mellitus (NDM)       1.00      1.00      1.00       811
                               Prediabetic       0.88      0.97      0.92       807
                           Type 1 Diabetes       0.87      0.89      0.88       817
                          Wolfram Syndrome       0.86      0.93      0.89       797
                                      LADA       0.91      0.86      0.89       784
                           Type 2 Diabetes       0.85      0.70      0.77       810
                 Wolcott-Rallison Syndrome       0.92      0.84      0.88       810
                        Second

# ENSEMBLING

wip